# Biblioteka do automatycznego różniczkowania

Kacper Aleksander 325446

In [16]:
abstract type Operator end # abstrakcyjny typ Operator
const Chain = Vector{Operator} # alias Chain do Vector{Operator}
abstract type Loss end # abstrakcyjny typ Loss

# Tensor
struct Tensor{N}
  # struktura Tensor z parametrem N, który jest przekazywany do NTuple. Czyli Tensor N
  # składa się z atrybutu outsize, który jest typu NTuple o długości N i każdy z jego
  # elementów ma typ Int64 
  outsize :: NTuple{N, Int64}
end
  tensor(sz...) = Tensor(sz)()
# ... to tzw. splat operator, jak *args w Pythonie. Można wtedy pisać tensor(2, 3, 4), co spowoduje spakowanie
# tych wartości do krotki: sz = (2, 3, 4) i uruchomienie konstruktora Tensor(sz). Krotka sz zostanie przekształcona
# na NTuple{N, Int64} automatycznie, aby dopasować się do definicji atrybutu "outsize" Tensora.

function (x::Tensor{N})() where N
  data = zeros(x.outsize...)
  return GraphNode(data)
end

function chain(operators)
# chain to funkcja tworząca wektor operatorów. Wejściem jest wektor składający się z pojedynczych
# operatorów lub krotek operatorów - w tym drugim przypadku są one odpakowywane
  function flatten(x::Tuple)
    y = Vector{Operator}()
    for v in x
      if v isa Tuple
        push!(y, v...)
      else
        push!(y, v)
      end
      end
    return y
  end

  result = Vector{Operator}()
  for operator in flatten(operators)
    push!(result, operator)
  end
  return result
end

function (chain::Chain)(x)
# (chain::Chain) oznacza, że jest to funkcja nieprzypisana do nazwy, a do każdego z obiektów
# typu Chain. x to argument.
  node = x # wartość początkowa
  for op in chain
    node = op(node) # dla każdej operacji w chainie (wektorze operacji) wykonaj tę operację na wartości node.
  end
  return node # zwróć uzyskaną wartość
end

In [ ]:
mutable struct GraphNode{OP, N}
  args :: NTuple{N, GraphNode}
  grad
  data
end

const GraphWeight = GraphNode{:weight, 0} # Węzeł bez argumentów gdzie OP = :weight
const GraphTensor = GraphNode{:tensor, 0} # Węzeł bez argumentów gdzie OP = :tensor

# Konstruktor dla danych jak wagi i tensory (obrazy), czyli nieposiadających argumentów (rodziców):
function GraphNode(data::T, trainable=false) where T
  if trainable # wagi są trainable
    return GraphNode{:weight, 0}((), zero(data), data)
  else # tensory nie są trainable
    return GraphNode{:tensor, 0}((), zero(data), data)
  end
end

# Konstruktor dla operatorów-symboli, czyli operatorów, które zaczynają się od :, np. :mul.
# Wymaga podania argumentów! (args)
function GraphNode(op::Symbol, args::Tuple, data::T) where T
  N = length(args)
  grad = similar(data)
  return GraphNode{op, N}(args, grad, data)
end

# Zwracanie wektora GraphNode'ów w kolejności:
function graph(node)
  function visit!(node::GraphNode, visited, ordered)
    if node in visited
    else # jeśli node nie był odwiedzony
      push!(visited, node) # dodaj node do odwiedzonych
      for arg in node.args # dla każdego argumentu w node (jeżeli są)
        visit!(arg, visited, ordered)
        # Odwiedź argumenty ZANIM dodasz obecny node do listy.
        # Gwarantuje to odpowiednią kolejność w grafie. 
      end
      push!(ordered, node)
    end
    return nothing
  end
  ordered = Vector{GraphNode}() # utwórz wektor gdzie każdy element to GraphNode
  visited = Set{GraphNode}() # utwórz wektor z odwiedzonymi GraphNode'ami
  visit!(node, visited, ordered) # odwiedź node i modyfikuj visited oraz ordered
  return ordered # zwróć uporządkowany wektor GraphNode'ów
end

# Funkcja do wyzerowywania gradientu w całym wektorze GraphNode'ów:
function zerograd!(order :: Vector{GraphNode})
  for node in order
    node.grad .= 0
  end
end

# Funkcje, które nic nie robią, ale zapobiegają błędowi kompilatora.
# primal! liczy wartość przejścia w przód - ale nie trzeba jej liczyć dla GraphTensor
# i GraphWeight, ponieważ te wartości są ustalone.
# adjoint! liczy gradient.
function primal!(tensor::GraphTensor) end
function primal!(weight::GraphWeight) end
function adjoint!(::GraphTensor) end
function adjoint!(::GraphWeight) end

# Funkcja forward! przyjmuje posortowany wektor GraphNode'ów oraz pary
# i inicjalizuje data w GraphNode'ach faktycznymi wartościami, a następnie
# przechodzi po Vector{GraphNode} i liczy wartości za pomocą funkcji określonych
# w sekcji Działanie operacji.
function forward!(order::Vector{GraphNode}, pairs...)
  for pair in pairs
    tensor,data = pair
    tensor.data .= data
  end

  for node in order
    primal!(node)
  end
end

# Funkcja backward! liczy gradient.
function backward!(order::Vector{GraphNode})
	seed = last(order)
	seed.grad .= 1

  for node in reverse(order) # w przeciwnym kierunku niż forward!
    adjoint!(node)
  end
end

# Algorytm spadku gradientu

function optimize!(graph, η)
  for node in graph
	if node isa GraphWeight
      node.data .-= η * node.grad
    end
  end
end

# Wyświetlanie

import Base: show
show(io::IO, x::GraphNode{OP, N}) where {OP,N} =
  print(io, "layer ", OP, " with ", N, " arg(s)")
show(io::IO, x::GraphWeight) = print(io, "weight")
show(io::IO, x::GraphTensor) = print(io, "tensor")

backward! (generic function with 3 methods)

## Operatory

In [20]:
using LinearAlgebra

# Sigmoid
struct Sigmoid <: Operator end # Sigmoid jest jednym z operatorów
sigmoid() = Sigmoid() # uproszczenie zapisu konstruktora (z małej litery)

function (y::Sigmoid)(x)
  sz = length(x.data)
  return GraphNode(:sigmoid, (x,), zeros(sz))
end

# ReLU
struct ReLU <: Operator end # ReLU jest jednym z Operatorów
relu() = ReLU() # uproszczenie zapisu konstruktora (z małej litery)

function (y::ReLU)(x)
  sz  = length(x.data)
  return GraphNode(:relu, (x,), zeros(sz))
end

# Dense
struct Dense <: Operator # Dense jest jednym z Operatorów
  insize  :: Int64
  outsize :: Int64
end

dense(pair :: Pair{Int64, Int64}) =
  Dense(first(pair), last(pair)) # umożliwia tworzenie warstwy Dense za pomocą pary: np. dense(2 => 16) wykonuje Dense(2, 16)
dense(pair :: Pair{Int64, Int64}, activation) =
  tuple(dense(pair), activation()) # umożliwia tworzenie warstwy wraz z funkcją aktywacji, np. dense(2 => 16, relu)

function (y::Dense)(x)
  n   = y.insize
  m   = y.outsize
  W   = GraphNode(randn(m, n), true) # macierz wag m x n 
  b   = GraphNode(randn(m), true) # jednowymiarowa macierz biasów o długości m
  mul = GraphNode(:mul, (W, x), zeros(m)) # wymnożenie x przez wagi
  add = GraphNode(:add, (mul, b), zeros(m)) # dodanie biasu
  return add
end

# BinaryCrossEntropy
struct BinaryCrossEntropy <: Loss end # BinaryCrossEntropy należy do Loss
bce(output, target) = BinaryCrossEntropy()(output, target) # skrócenie zapisu

function (E::BinaryCrossEntropy)(x, y)
  return GraphNode(:bce, (x, y), zeros(1))
end

## Działanie operacji

In [21]:
function primal!(z::GraphNode{:bce, 2})
  x, y = z.args
  z.data = -(y.data .* log.(x.data) + (1 .- y.data) .* log.(1 .- x.data))
  return nothing
end

function adjoint!(z::GraphNode{:bce, 2})
  x, y = z.args
  x.grad -= y.data ./ x.data .* z.grad
  x.grad += (1 .- y.data) ./ (1 .- x.data) .* z.grad
  return nothing
end

function primal!(y::GraphNode{:mul, 2})
  W, x = y.args
  y.data = W.data * x.data
  return nothing
end

function adjoint!(y::GraphNode{:mul, 2})
  W, x = y.args
  W.grad += y.grad * x.data'
  x.grad += W.data' * y.grad
  return nothing
end

function primal!(y::GraphNode{:relu, 1})
  x, = y.args
  y.data .= max.(0, x.data)
  return nothing
end

function adjoint!(y::GraphNode{:relu, 1})
  x, = y.args
  for i in 1:length(x.data)
    if x.data[i] == y.data[i]
      x.grad[i] += y.grad[i]
	end
  end
  return nothing
end

function primal!(z::GraphNode{:add, 2})
  x, y = z.args
  z.data = x.data .+ y.data
  return nothing
end

function adjoint!(z::GraphNode{:add, 2})
  x, y = z.args
  x.grad += z.grad
  y.grad += z.grad
  return nothing
end

function primal!(z::GraphNode{:dot, 2})
  x, y = z.args
  z.data = dot(x.data, y.data)
  return nothing
end

function adjoint!(z::GraphNode{:dot, 2})
  x, y = z.args
  x.grad += y.data .* z.grad
  y.grad += x.data .* z.grad
  return nothing
end

function primal!(y::GraphNode{:sum, 1})
  x, = y.args
  y.data = sum(x.data)
  return nothing
end

function adjoint!(y::GraphNode{:sum, 1})
  x, = y.args
  x.grad += y.grad
  return nothing
end

function primal!(y::GraphNode{:sigmoid, 1})
  x, = y.args
  y.data = 1 ./ (1 .+ exp.(-x.data))
  return nothing
end

function adjoint!(y::GraphNode{:sigmoid, 1})
  x, = y.args
  x.grad += exp.(-x.data) ./ (1 .+ exp.(-x.data)) .^ 2 .* y.grad
  return nothing
end

adjoint! (generic function with 27 methods)

## Uruchomienie biblioteki

In [22]:
using Random

# Zdefiniowanie architektury sieci:
net =
  chain((
		dense(2  => 16, relu),
		dense(16 => 1, sigmoid),
  ))

input  = tensor(2) # Wejście
target = tensor(1) # Rozmiar targetu
output = net(input) # Zwraca output (wynik)
loss   = bce(output, target) # Zwraca ostatni GraphNode, liczący loss
model  = graph(loss) # Zwraca wektor Vector{GraphNode}, który wskazuje kolejność GraphNode'ów
println("Order of GraphNodes: ", model)

function data(N)
  c = ([-1,-1], [-1,+1], [+1,-1], [+1,+1])
	y = (0, 1, 1, 0)
	xs = zeros(2, N)
	ys = zeros(N)

	for i in 1:N
    j = rand(1:4)
		xs[:,i] .= 0.1randn(2) + c[j]
		ys[i] = y[j]
	end

	return xs, ys
end

function test(model)
  for (x, y) in zip(([+1,+1], [+1,-1], [-1,+1], [-1,-1]), ([0],[1],[1],[0]))
    forward!(model, input => x, target => y)
    result = round(output.data[1], digits=3)
    print(x[1] > 0 ? "1" : "0", " xor ", x[2] > 0 ? "1" : "0", " = ")
	println(result)
  end
end

inputs, targets = data(1000)

function train!(model, batch, inputs, targets)
  shuffle!(batch)
  L = 0.0
  for sample in batch
    zerograd!(model)
	forward!(model,
			 input  => inputs[:,sample],
			 target => targets[sample,:])
    backward!(model)
	L += model[end].data[1]
	optimize!(model, 1e-1)
  end
	return L
end

batch = collect(1:500)
println("[x] Random model:")
test(model)
println("[x] Training...")
for _ in 1:5
  L = train!(model, batch, inputs, targets)
  println("[+] Loss: ", L)
end
println("[x] Final model:")
test(model)

Order of GraphNodes: GraphNode[weight, weight, tensor, layer mul with 2 arg(s), weight, layer add with 2 arg(s), layer relu with 1 arg(s), layer mul with 2 arg(s), weight, layer add with 2 arg(s), layer sigmoid with 1 arg(s), tensor, layer bce with 2 arg(s)]
[x] Random model:
1 xor 1 = 1.0
1 xor 0 = 0.998
0 xor 1 = 0.98
0 xor 0 = 0.828
[x] Training...
[+] Loss: 40.50886822589423
[+] Loss: 1.9712159690050528
[+] Loss: 1.0473899998457434
[+] Loss: 0.7037394252359106
[+] Loss: 0.5249335686877256
[x] Final model:
1 xor 1 = 0.001
1 xor 0 = 0.999
0 xor 1 = 0.999
0 xor 0 = 0.001
